# Машинное обучение с scikit-learn
Пример классификации ирисов

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 1. Загрузка данных

In [ ]:
# Загружаем известный датасет Iris
iris = load_iris()
X = iris.data
y = iris.target

# Создаём DataFrame для удобства
df = pd.DataFrame(X, columns=iris.feature_names)
df['target'] = y
df['species'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})
df.head(10)

In [ ]:
print('Размер данных:', X.shape)
print('Классы:', iris.target_names)
print('\nРаспределение классов:')
df['species'].value_counts()

## 2. Визуализация данных

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['red', 'green', 'blue']
for i, species in enumerate(iris.target_names):
    mask = y == i
    axes[0].scatter(X[mask, 0], X[mask, 1], c=colors[i], label=species, alpha=0.7)
    axes[1].scatter(X[mask, 2], X[mask, 3], c=colors[i], label=species, alpha=0.7)

axes[0].set_xlabel('Sepal Length')
axes[0].set_ylabel('Sepal Width')
axes[0].set_title('Чашелистики')
axes[0].legend()

axes[1].set_xlabel('Petal Length')
axes[1].set_ylabel('Petal Width')
axes[1].set_title('Лепестки')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Подготовка данных

In [ ]:
# Разделяем на обучающую и тестовую выборки (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Обучающая выборка: {X_train.shape[0]} примеров')
print(f'Тестовая выборка: {X_test.shape[0]} примеров')

In [ ]:
# Нормализация данных
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Обучение модели

In [ ]:
# Создаём и обучаем Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

print('Модель обучена!')

## 5. Оценка модели

In [ ]:
# Предсказания на тестовых данных
y_pred = model.predict(X_test_scaled)

# Точность
accuracy = accuracy_score(y_test, y_pred)
print(f'Точность модели: {accuracy * 100:.1f}%')

In [ ]:
# Детальный отчёт
print('Отчёт классификации:\n')
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# Матрица ошибок
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks([0, 1, 2], iris.target_names)
plt.yticks([0, 1, 2], iris.target_names)
plt.xlabel('Предсказано')
plt.ylabel('Фактически')
plt.title('Матрица ошибок')

for i in range(3):
    for j in range(3):
        plt.text(j, i, cm[i, j], ha='center', va='center', fontsize=16)

plt.tight_layout()
plt.show()

## 6. Важность признаков

In [ ]:
# Какие признаки важнее для модели?
importance = pd.DataFrame({
    'feature': iris.feature_names,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 5))
plt.barh(importance['feature'], importance['importance'], color='steelblue')
plt.xlabel('Важность')
plt.title('Важность признаков')
plt.tight_layout()
plt.show()

## 7. Предсказание для новых данных

In [ ]:
# Пример: новый цветок
new_flower = [[5.1, 3.5, 1.4, 0.2]]  # sepal_length, sepal_width, petal_length, petal_width

new_flower_scaled = scaler.transform(new_flower)
prediction = model.predict(new_flower_scaled)
probability = model.predict_proba(new_flower_scaled)

print(f'Предсказанный вид: {iris.target_names[prediction[0]]}')
print(f'\nВероятности:')
for i, name in enumerate(iris.target_names):
    print(f'  {name}: {probability[0][i]*100:.1f}%')